# Práctica 3: Fundamentos de Python para Análisis de Datos Científicos
**Programa de Preparación CERN 2026 — Semana 2**  
* **Estudiante:** Klever López  
* **Módulo:** [Software Carpentry — Programming with Python](https://swcarpentry.github.io/python-novice-inflammation/)  
* **Objetivo:** Cargar, explorar, manipular y visualizar matrices de datos clínicos bidimensionales utilizando las librerías `numpy` y `matplotlib`, aplicando buenas prácticas de ingeniería, programación modular y auditoría forense de datos.

## 1. Importación de Librerías y Carga del Dataset
Importamos las dependencias científicas organizadas alfabéticamente según la norma PEP 8 y cargamos el archivo `inflammation-01.csv` en un arreglo bidimensional continuo (`ndarray`).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Carga de datos tabulares delimitados por comas hacia memoria continua (C-order)
data = np.loadtxt(fname='data/inflammation-01.csv', delimiter=',')

# Inspección de propiedades estructurales del arreglo en memoria
print(f"Tipo de estructura: {type(data)}")
print(f"Dimensiones de la matriz (pacientes, días): {data.shape}")
print(f"Tipo de dato de los elementos: {data.dtype}")
print(f"Total de observaciones registradas: {data.size}")

## 2. Exploración Matricial y Rebanado Bidimensional (Slicing 2D)
Accedemos a subconjuntos de la matriz utilizando la sintaxis `data[filas, columnas]`. Recordar que Python utiliza indexación basada en cero y rangos semiabiertos `[inicio:fin)` donde el límite superior se excluye.

In [ ]:
# Acceso a una celda puntual: Paciente 0, Día 0
print(f"Valor inicial (Paciente 0, Día 0): {data[0, 0]}")

# Acceso a una coordenada central: Paciente 30, Día 20
print(f"Valor central (Paciente 30, Día 20): {data[30, 20]}")

# Extracción de una submatriz: primeros 3 pacientes (filas 0, 1, 2) y primeros 5 días (columnas 0 a 4)
subconjunto = data[0:3, 0:5]
print("\nSubmatriz de muestra (3 pacientes x primeros 5 días):")
print(subconjunto)

## 3. Agregación Estadística por Ejes (Axis Collapse)
En NumPy, el parámetro `axis` especifica la dimensión que colapsa la operación matemática:
* **`axis=0`:** Colapso vertical de las 60 filas $\rightarrow$ produce métricas para cada uno de los **40 días**.
* **`axis=1`:** Colapso horizontal de las 40 columnas $\rightarrow$ produce métricas para cada uno de los **60 pacientes**.

In [ ]:
# 1. Agregación a lo largo del eje 0 (comportamiento diario global)
promedio_diario = np.mean(data, axis=0)
maximo_diario = np.max(data, axis=0)
minimo_diario = np.min(data, axis=0)
desviacion_diaria = np.std(data, axis=0)

print(f"Vector de promedios diarios — Longitud: {promedio_diario.shape[0]} días")
print(f"Rango global de inflamación diaria: Mínimo = {minimo_diario.min()}, Máximo = {maximo_diario.max()}")

# 2. Agregación a lo largo del eje 1 (comportamiento individual por paciente)
promedio_pacientes = np.mean(data, axis=1)
print(f"Vector de promedios por paciente — Longitud: {promedio_pacientes.shape[0]} pacientes")

## 4. Visualización Global: Mapa de Calor (Heatmap)
Renderizamos la matriz completa de 2400 elementos mediante `plt.imshow` para identificar visualmente las fases del ensayo clínico.

In [ ]:
plt.figure(figsize=(9, 4.5))
plt.imshow(data, aspect='auto', cmap='viridis')
plt.colorbar(label='Nivel de Inflamación (Unidades Arbitrarias)')
plt.title('Mapa de Calor: Distribución Espacio-Temporal del Ensayo Clínico', fontsize=11, weight='bold')
plt.xlabel('Día de Tratamiento (0 a 39)', fontsize=10)
plt.ylabel('Índice de Paciente (0 a 59)', fontsize=10)
plt.show()

## 5. Visualización Técnica conforme a Estándares de Métodos Numéricos (ESPE)
Implementamos un panel múltiple de subplots (1 fila $\times$ 3 columnas) con:
1. **Título general de la figura** en la parte superior izquierda (`fig.suptitle`).
2. **Curvas continuas combinadas con marcadores discretos** (`marker='o'`, `marker='s'`, `marker='^'`).
3. **Doble cuadrícula** (grilla principal continua y secundaria punteada mediante `minorticks_on`).
4. **Etiquetas visibles en el último dato** con flechas de llamada (`annotate`) indicando el valor final alcanzado.

In [ ]:
# Configuración del lienzo general (15 pulgadas de ancho x 4.5 de alto)
fig = plt.figure(figsize=(15.0, 4.5))

# Título general superior izquierdo (Estándar ESPE)
fig.suptitle('Evaluación Estadística de Inflamación — Archivo 01', x=0.05, y=0.98, ha='left', fontsize=13, weight='bold')

# -------------------------------------------------------------------------
# Subplot 1: Promedio Diario de Inflamación
# -------------------------------------------------------------------------
axes1 = fig.add_subplot(1, 3, 1)
y1 = np.mean(data, axis=0)
axes1.plot(range(40), y1, linestyle='-', marker='o', markersize=3.5, color='royalblue', label='Promedio')
axes1.set_title('Promedio Diario', fontsize=11, weight='bold')
axes1.set_xlabel('Día de Tratamiento (Iteraciones)', fontsize=10)
axes1.set_ylabel('Nivel de Inflamación Promedio', fontsize=10)
axes1.set_xticks(np.arange(0, 41, 5))
axes1.minorticks_on()
axes1.grid(True, which='major', linestyle='-', alpha=0.6)
axes1.grid(True, which='minor', linestyle=':', alpha=0.3)

# Anotación técnica en el dato final (Día 39)
axes1.annotate(
    f'Final: {y1[-1]:.2f}',
    xy=(39, y1[-1]),
    xytext=(26, y1[-1] + 2.5),
    arrowprops={'arrowstyle': '->', 'color': 'navy', 'lw': 1.2},
    bbox={'boxstyle': 'round,pad=0.2', 'fc': 'lightcyan', 'ec': 'royalblue', 'lw': 1},
    fontsize=9,
    weight='bold'
)

# -------------------------------------------------------------------------
# Subplot 2: Máximo Diario (Rampa Lineal Sospechosa)
# -------------------------------------------------------------------------
axes2 = fig.add_subplot(1, 3, 2)
y2 = np.max(data, axis=0)
axes2.plot(range(40), y2, linestyle='-', marker='s', markersize=3.5, color='crimson', label='Máximo')
axes2.set_title('Máximo Diario (Rampa Lineal)', fontsize=11, weight='bold')
axes2.set_xlabel('Día de Tratamiento (Iteraciones)', fontsize=10)
axes2.set_ylabel('Nivel de Inflamación Máxima', fontsize=10)
axes2.set_xticks(np.arange(0, 41, 5))
axes2.set_yticks(np.arange(0, 21, 2))
axes2.minorticks_on()
axes2.grid(True, which='major', linestyle='-', alpha=0.6)
axes2.grid(True, which='minor', linestyle=':', alpha=0.3)

# Anotación técnica en el dato final (Día 39)
axes2.annotate(
    f'Final: {y2[-1]:.1f}',
    xy=(39, y2[-1]),
    xytext=(26, y2[-1] + 3.0),
    arrowprops={'arrowstyle': '->', 'color': 'darkred', 'lw': 1.2},
    bbox={'boxstyle': 'round,pad=0.2', 'fc': 'mistyrose', 'ec': 'crimson', 'lw': 1},
    fontsize=9,
    weight='bold'
)

# -------------------------------------------------------------------------
# Subplot 3: Mínimo Diario (Gradas Discretas de 4 días)
# -------------------------------------------------------------------------
axes3 = fig.add_subplot(1, 3, 3)
y3 = np.min(data, axis=0)
# Línea en escalón ortogonal con marcadores sobre los puntos muestreados
axes3.step(range(40), y3, where='mid', color='forestgreen', linewidth=1.5)
axes3.plot(range(40), y3, linestyle='none', marker='^', markersize=4, color='forestgreen', label='Mínimo')
axes3.set_title('Mínimo Diario (Escalones Discretos)', fontsize=11, weight='bold')
axes3.set_xlabel('Día de Tratamiento (Iteraciones)', fontsize=10)
axes3.set_ylabel('Nivel de Inflamación Mínima', fontsize=10)
axes3.set_xticks(np.arange(0, 41, 4))
axes3.set_yticks(np.arange(0, 6, 1))
axes3.minorticks_on()
axes3.grid(True, which='major', linestyle='-', alpha=0.6)
axes3.grid(True, which='minor', linestyle=':', alpha=0.3)

# Anotación técnica en el dato final (Día 39)
axes3.annotate(
    f'Final: {y3[-1]:.1f}',
    xy=(39, y3[-1]),
    xytext=(26, y3[-1] + 1.2),
    arrowprops={'arrowstyle': '->', 'color': 'darkgreen', 'lw': 1.2},
    bbox={'boxstyle': 'round,pad=0.2', 'fc': 'honeydew', 'ec': 'forestgreen', 'lw': 1},
    fontsize=9,
    weight='bold'
)

# Ajuste de márgenes preservando el espacio para el suptitle superior
fig.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

## 6. Funciones Modulares de Diagnóstico y Automatización
Empaquetamos la lógica analítica en funciones modulares reutilizables con docstrings normalizados (PEP 257) y diccionarios literales sin sobrecarga.

In [ ]:
def detectar_anomalias(data):
    """
    Audita la matriz de datos para detectar comportamientos matemáticos anómalos.
    
    Parámetros:
        data (np.ndarray): Matriz bidimensional de datos clínicos (pacientes x días).
    """
    max_diario = np.max(data, axis=0)
    min_diario = np.min(data, axis=0)
    
    # Condición 1: Rampa lineal artificial en máximos
    if max_diario[0] == 0 and max_diario[20] == 20:
        print('  ⚠️  ALERTA: Los valores máximos forman una rampa lineal artificial.')
    # Condición 2: Sensores o registros bloqueados en cero
    elif np.sum(min_diario) == 0:
        print('  ⚠️  ALERTA: Los valores mínimos son cero todos los días (posible falla de sensor).')
    else:
        print('  ✅  COMPORTAMIENTO: Los datos parecen biológicamente naturales.')


def analizar(filename):
    """
    Carga un archivo CSV, ejecuta la detección de anomalías y genera el panel de subplots.
    
    Parámetros:
        filename (str): Ruta al archivo tabular CSV.
    """
    print(f"\n{'=' * 50}")
    print(f"AUDITORÍA CLÍNICA: {filename}")
    print(f"{'=' * 50}")
    
    # Carga de datos
    data = np.loadtxt(fname=filename, delimiter=',')
    detectar_anomalias(data)
    
    # Generación de gráficos técnicos
    fig = plt.figure(figsize=(15.0, 4.2))
    fig.suptitle(f'Estudio Clínico: {filename}', x=0.05, y=0.98, ha='left', fontsize=12, weight='bold')
    
    # Panel 1: Promedio
    ax1 = fig.add_subplot(1, 3, 1)
    y1 = np.mean(data, axis=0)
    ax1.plot(range(40), y1, linestyle='-', marker='o', markersize=3, color='royalblue')
    ax1.set_title('Promedio Diario', fontsize=10, weight='bold')
    ax1.set_xlabel('Día de Tratamiento', fontsize=9)
    ax1.set_ylabel('Inflamación', fontsize=9)
    ax1.minorticks_on()
    ax1.grid(True, which='major', linestyle='-', alpha=0.5)
    ax1.grid(True, which='minor', linestyle=':', alpha=0.25)
    ax1.annotate(
        f'{y1[-1]:.2f}',
        xy=(39, y1[-1]),
        xytext=(28, y1[-1] + 2.0),
        arrowprops={'arrowstyle': '->', 'color': 'navy'},
        bbox={'boxstyle': 'round,pad=0.2', 'fc': 'white', 'ec': 'royalblue', 'lw': 0.8},
        fontsize=8
    )
    
    # Panel 2: Máximo
    ax2 = fig.add_subplot(1, 3, 2)
    y2 = np.max(data, axis=0)
    ax2.plot(range(40), y2, linestyle='-', marker='s', markersize=3, color='crimson')
    ax2.set_title('Máximo Diario', fontsize=10, weight='bold')
    ax2.set_xlabel('Día de Tratamiento', fontsize=9)
    ax2.minorticks_on()
    ax2.grid(True, which='major', linestyle='-', alpha=0.5)
    ax2.grid(True, which='minor', linestyle=':', alpha=0.25)
    ax2.annotate(
        f'{y2[-1]:.1f}',
        xy=(39, y2[-1]),
        xytext=(28, y2[-1] + 3.0),
        arrowprops={'arrowstyle': '->', 'color': 'darkred'},
        bbox={'boxstyle': 'round,pad=0.2', 'fc': 'white', 'ec': 'crimson', 'lw': 0.8},
        fontsize=8
    )
    
    # Panel 3: Mínimo
    ax3 = fig.add_subplot(1, 3, 3)
    y3 = np.min(data, axis=0)
    ax3.step(range(40), y3, where='mid', color='forestgreen')
    ax3.plot(range(40), y3, linestyle='none', marker='^', markersize=3.5, color='forestgreen')
    ax3.set_title('Mínimo Diario', fontsize=10, weight='bold')
    ax3.set_xlabel('Día de Tratamiento', fontsize=9)
    ax3.minorticks_on()
    ax3.grid(True, which='major', linestyle='-', alpha=0.5)
    ax3.grid(True, which='minor', linestyle=':', alpha=0.25)
    ax3.annotate(
        f'{y3[-1]:.1f}',
        xy=(39, y3[-1]),
        xytext=(28, y3[-1] + 1.0),
        arrowprops={'arrowstyle': '->', 'color': 'darkgreen'},
        bbox={'boxstyle': 'round,pad=0.2', 'fc': 'white', 'ec': 'forestgreen', 'lw': 0.8},
        fontsize=8
    )
    
    fig.tight_layout(rect=[0, 0, 1, 0.94])
    plt.show()

## 7. Automatización por Lotes con `glob`
Utilizamos la librería estándar `glob` para capturar dinámicamente los archivos que coincidan con el patrón `data/inflammation-*.csv` y procesarlos en bucle de forma determinista mediante `sorted()`.

In [ ]:
import glob

# Búsqueda y ordenamiento lexicográfico de archivos
archivos_dataset = sorted(glob.glob('data/inflammation-*.csv'))
print(f"Total de archivos disponibles en el directorio: {len(archivos_dataset)}")

# Procesamiento sistemático de una muestra de control (primeros 3 archivos)
for ruta_archivo in archivos_dataset[:3]:
    analizar(ruta_archivo)

## 8. Verificación Rigurosa de Linealidad con `np.diff()`
Para trascender la comprobación puntual de extremos (`max[20] == 20`), calculamos la derivada discreta (la tasa de cambio día a día) sobre la fase de ascenso. Demostramos que la pendiente es unitaria y constante en cada iteración.

In [ ]:
# Extracción de máximos del archivo 01
data_01 = np.loadtxt(fname='data/inflammation-01.csv', delimiter=',')
maximos_01 = np.max(data_01, axis=0)

# Cálculo de diferencias consecutivas (derivada discreta) durante los primeros 20 días
diferencias_ascenso = np.diff(maximos_01[:21])
print(f"Derivada discreta (días 0 a 20): {diferencias_ascenso}")

# Verificación booleana estricta
es_rampa_matematica = np.all(diferencias_ascenso == 1.0)
print(f"¿Pendiente rigurosamente constante e idéntica a 1.0?: {es_rampa_matematica}")

## 9. Conclusiones Técnicas y Científicas

1. **Eficiencia en Memoria:** Los objetos `ndarray` de NumPy proveen almacenamiento contiguo en memoria RAM, permitiendo operaciones vectorizadas a bajo nivel (C) que eliminan la necesidad de bucles lentos en Python puro.
2. **Dominio de Agregación Direccional:** El control explícito sobre `axis=0` (colapso de filas de pacientes) y `axis=1` (colapso de columnas temporales) garantiza cálculos matriciales reproducibles y libres de ambigüedades.
3. **Estándares de Gráficos de Ingeniería:** La inclusión de marcadores combinados con líneas continuas, doble cuadrícula milimétrica (`major`/`minor`) y anotaciones numéricas en el dato final (`annotate`) eleva la calidad visual de las figuras a estándares rigurosos de publicación técnica.
4. **Auditoría Forense de Datos:** La combinación de derivadas discretas (`np.diff`) y análisis de escalones ortogonales (`step`) evidenció de manera concluyente el origen artificial y sintético de los datasets analizados (`inflammation-01.csv` e `inflammation-02.csv`), así como fallas instrumentales en `inflammation-03.csv`.